# 3.4 — Racinisation et lemmatisation

**Traitement automatique du langage naturel (TALN)**

## Objectifs pédagogiques
À la fin de ce notebook, vous devriez pouvoir :
- Expliquer la différence entre la racinisation (*stemming*) et la lemmatisation.
- Appliquer les algorithmes Porter, Snowball et Lancaster avec NLTK.
- Obtenir des lemmes avec spaCy et NLTK.
- Comprendre l’importance de la langue et de la catégorie grammaticale.
- Reconnaître les limites de l’application d’outils anglais à des textes arabes.

> **Remarque :** la racinisation et la lemmatisation sont deux techniques de normalisation différentes. Une racine produite par un stemmer n’est pas nécessairement un mot du dictionnaire ; un lemme vise une forme lexicale canonique.

## 1. Préparation
Installez les bibliothèques nécessaires si elles ne sont pas déjà disponibles. Exécutez la cellule d’installation uniquement si les packages sont absents.

In [ ]:
%pip install -q nltk spacy

Téléchargez le modèle anglais de spaCy et les données WordNet de NLTK utilisées plus loin. Une connexion Internet est nécessaire.

In [1]:
import nltk
import spacy

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

# Run this command if the English spaCy model is not installed:
# If needed, run in a separate notebook cell: !python -m spacy download en_core_web_sm

True

In [2]:
# Load the English model
nlp = spacy.load("en_core_web_sm")
print("spaCy model loaded successfully.")

spaCy model loaded successfully.


## 2. Racinisation : principe
Un stemmer supprime ou transforme les terminaisons selon des règles. Il n’analyse généralement pas tout le contexte grammatical ; le résultat peut donc ne pas être un mot valide.

In [3]:
from nltk.stem import PorterStemmer

porter = PorterStemmer()
words = ["run", "runner", "running", "ran", "runs", "easily", "fairly"]

for word in words:
    print(f"{word:12} -> {porter.stem(word)}")

run          -> run
runner       -> runner
running      -> run
ran          -> ran
runs         -> run
easily       -> easili
fairly       -> fairli


**À observer :** des mots proches peuvent être ramenés à la même forme, tandis que des formes irrégulières comme `ran` ne sont pas nécessairement ramenées à `run`. La racinisation repose sur des règles et ne constitue pas une compréhension complète du sens.

## 3. Comparer les stemmers courants de NLTK
- **PorterStemmer :** très utilisé, racinisation modérée.
- **SnowballStemmer :** famille de stemmers améliorés et multilingues.
- **LancasterStemmer :** souvent plus agressif ; il peut produire des formes plus courtes.

In [ ]:
from nltk.stem import PorterStemmer, LancasterStemmer, SnowballStemmer

porter = PorterStemmer()
lancaster = LancasterStemmer()
snowball = SnowballStemmer("english")

words = ["generous", "generation", "generously", "generate", "connect", "connected", "connection"]

print(f"{'Word':18}{'Porter':18}{'Lancaster':18}{'Snowball':18}")
print("-" * 72)
for word in words:
    print(f"{word:18}{porter.stem(word):18}{lancaster.stem(word):18}{snowball.stem(word):18}")

Les algorithmes peuvent produire des formes différentes pour un même mot. Le résultat le plus court n’est pas automatiquement le meilleur ; le choix dépend de la tâche et de la langue.

## 4. Racinisation dans un contexte
Découpez une phrase en tokens, puis appliquez le stemmer à chaque token. Selon le tokenizer, la ponctuation peut constituer des tokens séparés.

In [ ]:
from nltk.tokenize import word_tokenize

sentence = (
    "Had you booked the air booking yet? If not, try to book it ASAP; "
    "the booking will be out of books."
)
tokens = word_tokenize(sentence)

print(f"{'Token':18}{'Porter stem':18}{'Lancaster stem':18}")
print("-" * 54)
for token in tokens:
    print(f"{token:18}{porter.stem(token):18}{lancaster.stem(token):18}")

## 5. Lemmatisation : principe
La lemmatisation vise à retrouver la forme de dictionnaire (lemme). Contrairement à la racinisation simple, elle peut s’appuyer sur des informations lexicales et sur la catégorie grammaticale. Le POS est important : un mot peut avoir des lemmes différents selon qu’il est employé comme nom ou comme verbe.

### 5.1 Lemmatisation avec spaCy
Le modèle anglais fournit les catégories grammaticales et les lemmes. Dans spaCy, `token.lemma_` contient le lemme sous forme de chaîne lisible ; `token.lemma` est un identifiant numérique interne.

In [ ]:
doc = nlp("I am a runner running in a race because I love to run after I ran yesterday.")

print(f"{'Token':15}{'POS':10}{'Lemma':15}")
print("-" * 40)
for token in doc:
    print(f"{token.text:15}{token.pos_:10}{token.lemma_:15}")

In [ ]:
def show_lemmas(text):
    doc = nlp(text)
    print(f"{'Token':15}{'POS':10}{'Lemma':20}")
    print("-" * 45)
    for token in doc:
        print(f"{token.text:15}{token.pos_:10}{token.lemma_:20}")

show_lemmas("I saw eighteen mice today!")
show_lemmas("I am meeting him right now at the meeting.")

**À discuter :** `meeting` peut être une forme verbale ou un nom. Le contexte et l’étiquetage grammatical aident le lemmatiseur à choisir un lemme approprié.

In [ ]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

print(lemmatizer.lemmatize("meeting"))
# meeting (par défaut : nom)

print(lemmatizer.lemmatize("meeting", pos='v'))
# meet (verb

### 5.2 Lemmatisation avec WordNet dans NLTK
WordNetLemmatizer utilise par défaut la catégorie nom (`pos='n'`). Pour les verbes, indiquez `pos='v'`. Les résultats dépendent de la couverture de WordNet et de la catégorie grammaticale fournie.

In [ ]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
words = ["cats", "cacti", "radii", "feet", "speech", "runner"]

print(f"{'Word':15}{'Noun lemma (default)':25}")
print("-" * 40)
for word in words:
    print(f"{word:15}{lemmatizer.lemmatize(word):25}")

In [ ]:
print("meeting as a noun:", lemmatizer.lemmatize("meeting", pos="n"))
print("meeting as a verb:", lemmatizer.lemmatize("meeting", pos="v"))

words = ["is", "was", "be", "been", "are", "were"]
for word in words:
    print(f"{word:8} noun={lemmatizer.lemmatize(word, pos='n'):8} verb={lemmatizer.lemmatize(word, pos='v')}")

Comparez le comportement par défaut pour les noms à la lemmatisation des verbes. Pour obtenir des résultats fiables, une application réelle doit identifier la catégorie grammaticale au lieu de supposer que tous les tokens sont des noms.

### 5.3 Lemmatiser une phrase
L’exemple ci-dessous retire la ponctuation avec une compréhension de liste (au lieu de modifier une liste pendant son parcours). Il affiche les lemmes par défaut des noms et ceux des verbes.

In [ ]:
sentence = (
    "He was running and eating at the same time. "
    "He has a bad habit of swimming after playing long hours in the sun."
)
tokens = word_tokenize(sentence)
tokens_without_punctuation = [token for token in tokens if any(ch.isalnum() for ch in token)]

print(f"{'Word':18}{'Noun lemma':18}{'Verb lemma':18}")
print("-" * 54)
for word in tokens_without_punctuation:
    print(f"{word:18}{lemmatizer.lemmatize(word):18}{lemmatizer.lemmatize(word, pos='v'):18}")

## 6. Outils anglais et textes arabes
PorterStemmer et LancasterStemmer sont conçus pour l’anglais et ne doivent pas être considérés comme des stemmers arabes. Le SnowballStemmer de NLTK prend en charge l’arabe ; ses résultats doivent néanmoins être évalués selon la tâche visée. WordNetLemmatizer et le modèle anglais de spaCy ne sont pas des lemmatiseurs arabes.

In [ ]:
arabic_words = ["الجري", "تجري", "يجرون", "جري", "يجري"]

arabic_stemmer = SnowballStemmer("arabic")
print(f"{'Arabic word':18}{'Snowball stem':18}")
print("-" * 36)
for word in arabic_words:
    print(f"{word:18}{arabic_stemmer.stem(word):18}")

N’interprétez pas le résultat d’un stemmer anglais ou du lemmatiseur WordNet appliqué à l’arabe comme une analyse morphologique arabe pertinente. Pour le TALN arabe, choisissez un outil conçu et évalué spécifiquement pour cette langue.

## 7. Racinisation vs lemmatisation
| Aspect | Racinisation (*stemming*) | Lemmatisation |
|---|---|---|
| Méthode | Suppression/transformation par règles | Analyse lexicale, souvent sensible au POS |
| Résultat | Racine ; pas nécessairement un mot | Lemme proche de la forme dictionnaire |
| Contexte/POS | Généralement limité | Souvent important |
| Coût | Généralement plus rapide/simple | Souvent plus coûteuse en calcul |
| Usages | Recherche, indexation, normalisation approximative | Analyse linguistique et tâches nécessitant des lemmes |

## 8. Exercices
1. Appliquez Porter, Lancaster et Snowball à `studies`, `studying`, `studied`, `study`, puis comparez les résultats.
2. Utilisez spaCy pour afficher les tokens, les POS et les lemmes de : `The children were playing and then played outside.`
3. Appliquez WordNetLemmatizer à `saw`, `meeting` et `running` avec les catégories nom et verbe. Expliquez les différences.
4. Pourquoi `WordNetLemmatizer().lemmatize('running')` ne renvoie-t-il pas nécessairement `run` ?
5. Exécutez le stemmer Snowball arabe sur les exemples arabes. Expliquez pourquoi un stemmer anglais ne convient pas à ces mots.

## À retenir
- La racinisation et la lemmatisation ne sont pas interchangeables.
- Porter, Lancaster et Snowball peuvent produire des racines différentes.
- spaCy expose les lemmes avec `token.lemma_`.
- La lemmatisation WordNet de NLTK dépend du POS ; la catégorie par défaut est le nom.
- Utilisez toujours des outils adaptés à la langue traitée.